# **Purpose**

**Track C — LLM prompting (no fine-tuning).**

This notebook generates distractors purely via **prompting** a general-purpose
instruction-tuned LLM (`meta-llama/Llama-3.2-3B-Instruct`) — no training happens here. It runs
the *same* model under three prompting strategies (zero-shot, few-shot,
misconception-guided chain-of-thought) so the comparison isolates the effect
of the prompt design, not the backbone.

It scores on a deterministic **sub-sample** of the same fixed RACE test split
used by Track A and Track B (same `TEST_SIZE`/`SEED`), since LLM generation
is much slower per-item than the fine-tuned models or the similarity
baselines. Predictions for all 3 strategies are saved into a single JSON that
the matching evaluation notebook consumes.

**This is a gated model.** Before running this notebook:
1. Visit https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct and accept the license/usage terms.
2. Make sure the `HF_TOKEN` you add to Kaggle Secrets belongs to the same
   account that accepted that license.

**Runtime:** these are small (1.1B–3.8B parameter) models, so unlike larger
LLMs they load comfortably in fp16 on a T4 GPU with no quantization needed —
GPU is still recommended for speed, but there's no `bitsandbytes` dependency
here. Turn **Internet ON** in Kaggle's Settings panel.

## **Install dependencies**

In [1]:
!pip install -qU transformers accelerate datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.2 MB/s eta 0:00:00


## **Load the API Keys and Tokens**

In [2]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name you set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Imports**

In [3]:
import json
import random

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


## **Config**

`TEST_SIZE`/`SEED` must match Track A/Track B exactly. `LLM_EVAL_SIZE`
controls how many of those same fixed items are actually sent to the LLM —
these models are small enough that you can likely push this higher than you
would for a 7B+ model; lower it if a run is taking too long for your Kaggle
session budget.

In [4]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"           # swap this to compare TinyLlama / Qwen2.5 / Llama-3.2 / Phi-3.5
TEST_SIZE = 1500                        # keep identical to Track A/B
SEED = 42                                # keep identical to Track A/B
LLM_EVAL_SIZE = 300                      # sub-sample actually sent to the LLM (x3 strategies)

MAX_INPUT_TOKENS = 1024
BATCH_SIZE = 5                            # these are small models; raise further if you have headroom

OUTPUT_JSON = "/kaggle/working/llm_llama32_predictions.json"

## **Load the fixed test split**

RACE gives 3 gold distractors per question, so we first expand the full test
set into individual (context, question, correct answer) → gold-distractor
pairs, then shuffle those pairs with a fixed seed and take the first
`TEST_SIZE`. **Keep `TEST_SIZE`/`SEED` identical to Track A and Track B** —
that's what makes every method directly comparable.

LLM prompting is far slower per-item than the fine-tuned models or the
similarity baselines (large model, 3 prompting strategies per item), so we
then take a smaller, deterministic **sub-sample** of `LLM_EVAL_SIZE` items
from within that same fixed set for actual generation — same underlying
pool, just fewer of them, to keep runtime reasonable.

In [5]:
LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}

raw = load_dataset("race", "all")
test_examples = raw["test"]

flat_contexts, flat_questions, flat_answers, flat_distractors = [], [], [], []
for ex in test_examples:
    answer_letter = ex["answer"]
    if answer_letter not in LETTER_TO_IDX:
        continue
    correct_idx = LETTER_TO_IDX[answer_letter]
    options = ex["options"]
    if correct_idx >= len(options):
        continue
    correct_answer = options[correct_idx]

    for i, option in enumerate(options):
        if i == correct_idx or not option.strip():
            continue
        flat_contexts.append(ex["article"])
        flat_questions.append(ex["question"])
        flat_answers.append(correct_answer)
        flat_distractors.append(option)

print(f"Total (context, question, answer) -> gold distractor pairs available: {len(flat_distractors)}")

indices = list(range(len(flat_distractors)))
random.Random(SEED).shuffle(indices)
indices = indices[:TEST_SIZE]

contexts = [flat_contexts[i] for i in indices]
questions = [flat_questions[i] for i in indices]
correct_answers = [flat_answers[i] for i in indices]
gold_distractors = [flat_distractors[i] for i in indices]

print(f"Fixed comparison pool: {len(gold_distractors)} pairs")

# Deterministic sub-sample actually used for (slow) LLM generation
llm_contexts = contexts[:LLM_EVAL_SIZE]
llm_questions = questions[:LLM_EVAL_SIZE]
llm_correct_answers = correct_answers[:LLM_EVAL_SIZE]
llm_gold_distractors = gold_distractors[:LLM_EVAL_SIZE]

print(f"Sub-sampled for LLM generation: {len(llm_gold_distractors)} pairs")

README.md: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/37.4M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4934 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/87866 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4887 [00:00<?, ? examples/s]

Total (context, question, answer) -> gold distractor pairs available: 14802
Fixed comparison pool: 1500 pairs
Sub-sampled for LLM generation: 300 pairs


## **Load the model**

No quantization needed at this size — plain fp16 (or fp32 on CPU) is enough.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for correct batched generation with causal LMs

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()
print(f"Loaded {MODEL_NAME}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.2-3B-Instruct


## **Prompting strategies**

Three strategies, applied to the *same* model, so the comparison isolates
the effect of the prompt rather than the backbone:

- **Zero-shot** — a single direct instruction, no examples.
- **Few-shot** — the same instruction, preceded by 3 short illustrative
  question → distractor demonstrations (generic, domain-neutral examples —
  not drawn from RACE itself, so nothing about the test set leaks into the
  prompt).
- **Misconception-CoT** — asks the model to first name a plausible student
  misconception, then phrase that misconception as the distractor. We parse
  out just the final `Distractor:` line afterwards.

Every prompt ends with an explicit output-format instruction so the parsed
result is just the distractor text, not commentary.

In [7]:
import re

FEW_SHOT_EXAMPLES = [
    ("What is the capital of France?", "Paris", "Lyon"),
    ("Which gas do plants absorb during photosynthesis?", "Carbon dioxide", "Oxygen"),
    ("Who wrote the play Hamlet?", "William Shakespeare", "Christopher Marlowe"),
]

def task_prompt(context, question, answer):
    return (
        "You are an experienced teacher setting a multiple-choice exam question.\n\n"
        f"Context: {context}\n"
        f"Question: {question}\n"
        f"Correct Answer: {answer}\n\n"
        "Write ONE plausible but incorrect answer option (a distractor) for this question. "
        "The distractor should be clearly wrong but related to the topic, so a student with a "
        "misconception might choose it.\n\n"
        "Respond with ONLY the distractor text and nothing else \u2014 no explanation, no labels."
    )

def cot_prompt(context, question, answer):
    return (
        "You are an experienced teacher setting a multiple-choice exam question.\n\n"
        f"Context: {context}\n"
        f"Question: {question}\n"
        f"Correct Answer: {answer}\n\n"
        "Think step by step:\n"
        "1. Identify ONE common misconception or plausible-but-wrong idea a student might have "
        "that relates to this question.\n"
        "2. Phrase that misconception as a single, concise distractor option (similar length and "
        "style to the correct answer).\n\n"
        "Respond in EXACTLY this format:\n"
        "Misconception: <one sentence>\n"
        "Distractor: <the distractor text only>"
    )

def build_messages(strategy, context, question, answer):
    if strategy == "zero_shot":
        return [{"role": "user", "content": task_prompt(context, question, answer)}]

    if strategy == "few_shot":
        messages = []
        for ex_question, ex_answer, ex_distractor in FEW_SHOT_EXAMPLES:
            messages.append({
                "role": "user",
                "content": task_prompt("(general knowledge question, no passage needed)", ex_question, ex_answer),
            })
            messages.append({"role": "assistant", "content": ex_distractor})
        messages.append({"role": "user", "content": task_prompt(context, question, answer)})
        return messages

    if strategy == "misconception_cot":
        return [{"role": "user", "content": cot_prompt(context, question, answer)}]

    raise ValueError(f"Unknown strategy: {strategy}")

def extract_distractor(raw_text, strategy):
    if strategy == "misconception_cot":
        match = re.search(r"Distractor:\s*(.+)", raw_text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip().split("\n")[0].strip()
        return raw_text.strip()
    return raw_text.strip().split("\n")[0].strip()

STRATEGIES = ["zero_shot", "few_shot", "misconception_cot"]

## **Batched generation helper**

In [8]:
@torch.no_grad()
def generate_batch(prompts, max_new_tokens=64):
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    results = []
    input_len = inputs["input_ids"].shape[1]
    for i in range(len(prompts)):
        new_tokens = output_ids[i][input_len:]
        results.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return results

## **Generate distractors for all 3 strategies**

Loops over each strategy, builds a chat-formatted prompt per item (via
`apply_chat_template`, which handles the model-specific chat format
automatically), and generates in batches. Misconception-CoT gets a larger
`max_new_tokens` budget since it has to write the reasoning step before the
final `Distractor:` line.

In [9]:
all_records = []

for strategy in STRATEGIES:
    print(f"=== Generating with strategy: {strategy} ===")

    prompts = []
    for context, question, answer in zip(llm_contexts, llm_questions, llm_correct_answers):
        messages = build_messages(strategy, context, question, answer)
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompts.append(prompt_text)

    max_new = 96 if strategy == "misconception_cot" else 48
    strategy_predictions = []

    for start in range(0, len(prompts), BATCH_SIZE):
        batch = prompts[start:start + BATCH_SIZE]
        raw_outputs = generate_batch(batch, max_new_tokens=max_new)
        cleaned = [extract_distractor(t, strategy) for t in raw_outputs]
        strategy_predictions.extend(cleaned)
        print(f"  {min(start + BATCH_SIZE, len(prompts))}/{len(prompts)} done")

    for i in range(len(llm_correct_answers)):
        all_records.append({
            "strategy": strategy,
            "question": llm_questions[i],
            "correct_answer": llm_correct_answers[i],
            "gold_distractor": llm_gold_distractors[i],
            "generated_distractor": strategy_predictions[i],
        })

print(f"\nGenerated {len(all_records)} total records "
      f"({len(llm_correct_answers)} items x {len(STRATEGIES)} strategies)")

=== Generating with strategy: zero_shot ===


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  5/300 done
  10/300 done
  15/300 done
  20/300 done
  25/300 done
  30/300 done
  35/300 done
  40/300 done
  45/300 done
  50/300 done
  55/300 done
  60/300 done
  65/300 done
  70/300 done
  75/300 done
  80/300 done
  85/300 done
  90/300 done
  95/300 done
  100/300 done
  105/300 done
  110/300 done
  115/300 done
  120/300 done
  125/300 done
  130/300 done
  135/300 done
  140/300 done
  145/300 done
  150/300 done
  155/300 done
  160/300 done
  165/300 done
  170/300 done
  175/300 done
  180/300 done
  185/300 done
  190/300 done
  195/300 done
  200/300 done
  205/300 done
  210/300 done
  215/300 done
  220/300 done
  225/300 done
  230/300 done
  235/300 done
  240/300 done
  245/300 done
  250/300 done
  255/300 done
  260/300 done
  265/300 done
  270/300 done
  275/300 done
  280/300 done
  285/300 done
  290/300 done
  295/300 done
  300/300 done
=== Generating with strategy: few_shot ===
  5/300 done
  10/300 done
  15/300 done
  20/300 done
  25/300 done
  30/300

## **Save predictions for the evaluation notebook**

In [10]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(
        {
            "method": "llm_prompting",
            "model_name": MODEL_NAME,
            "test_size": TEST_SIZE,
            "seed": SEED,
            "llm_eval_size": LLM_EVAL_SIZE,
            "strategies": STRATEGIES,
            "records": all_records,
        },
        f,
        indent=2,
    )

print(f"Saved {len(all_records)} records to {OUTPUT_JSON}")

Saved 900 records to /kaggle/working/llm_llama32_predictions.json


## **Inspect sample generations per strategy**

In [11]:
for strategy in STRATEGIES:
    print(f"\n=== {strategy} ===")
    strategy_records = [r for r in all_records if r["strategy"] == strategy][:3]
    for r in strategy_records:
        print(f"Correct answer:       {r['correct_answer']}")
        print(f"Gold distractor:      {r['gold_distractor']}")
        print(f"Generated distractor: {r['generated_distractor']}")
        print("-" * 80)


=== zero_shot ===
Correct answer:       Working or talking with students.
Gold distractor:      Having a basketball game.
Generated distractor: Taking a nap in the staff room.
--------------------------------------------------------------------------------
Correct answer:       we must know who we are
Gold distractor:      we should know what we do
Generated distractor: We must always be willing to take risks to achieve our goals.
--------------------------------------------------------------------------------
Correct answer:       the high quality education and research and the wide range of courses
Gold distractor:      the convenient traffic
Generated distractor: The main reasons for Coteborg University's popularity according to the passage is the high standard of student accommodation.
--------------------------------------------------------------------------------

=== few_shot ===
Correct answer:       Working or talking with students.
Gold distractor:      Having a basketball g

## **Next steps**

- Run `evaluate_llama32_llm_race_distractor.ipynb` to score these
  predictions per-strategy with the same metric suite used for Track A/B.
- Try a Knowledge-Graph-augmented variant of the misconception-CoT prompt
  (retrieve related ConceptNet nodes for the correct answer and inject them
  as "reasoning paths" before asking for the distractor) as a 4th strategy.
- Raise `LLM_EVAL_SIZE` toward the full 1500 if your Kaggle session budget
  allows, for tighter confidence in the metric comparison.